<a href="https://colab.research.google.com/github/Everlyne-Nasimiyu/NASA-API-Data-Explorer/blob/main/API_checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The objective  is to practice accessing and using NASA's public APIs to retrieve and manipulate space-related data, including obtaining an API key, making API requests, processing data into a pandas DataFrame, and exporting the results to a CSV file for sharing.

In [ ]:
import requests

api_key = "A8aLr93b18zjr9HlwwGzgRhnOGfV5bTPmcswq07m"

url = "https://api.nasa.gov/planetary/apod"
params = {"api_key": api_key}

response = requests.get(url, params=params)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    data = response.json()  # convert the response to JSON

    # check the json keys to find image url
    print(data.keys())
    # usually contains 'url', 'title', 'explanation'
    image_url = data['url']
    print("Title:", data['title'])
    print("Image URL:", image_url)

    # display the image
    from IPython.display import Image
    display(Image(url=image_url))
else:
    print(f"Error: Failed to fetch data from NASA API. Status code: {response.status_code}")
    print(f"Response: {response.text}")

dict_keys(['copyright', 'date', 'explanation', 'hdurl', 'media_type', 'service_version', 'title', 'url'])
Title: Total Lunar Eclipse over Tsé Bit'a'í
Image URL: https://apod.nasa.gov/apod/image/2603/EclipseSequence_Murata_1080.jpg


In [ ]:
import pandas as pd
import requests
from datetime import datetime, timedelta

# Get today's date and 7 days ago for the API call
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

url = "https://api.nasa.gov/neo/rest/v1/feed?"
params = {
    "start_date": start_date,
    "end_date": end_date,
    "api_key": api_key # Use the api_key variable defined in the previous cell
  }
response = requests.get(url, params=params)

# Check if the request was successful before proceeding
if response.status_code == 200:
    asteroids_data = response.json()

    asteroids_list = [] # intialize an empty list to store asteroid information
    # Iterate through each date and then each asteroid for that date
    for date, asteroids in asteroids_data['near_earth_objects'].items():
        for asteroid in asteroids:
            asteroid_id = asteroid['id']
            name = asteroid['name']
            diameter_min = asteroid['estimated_diameter']['kilometers']['estimated_diameter_min']
            abs_mag = asteroid['absolute_magnitude_h']
            # Ensure 'close_approach_data' exists and is not empty
            rel_velocity = None
            if asteroid['close_approach_data']:
                rel_velocity = asteroid['close_approach_data'][0]['relative_velocity']['kilometers_per_second']

            asteroids_list.append({
                "Asteroid ID": asteroid_id,
                "Asteroid Name": name,
                "Minimal Diameter (km)": diameter_min,
                "Absolute Magnitude": abs_mag,
                "Relative Velocity (km/s)": rel_velocity
            })

    df = pd.DataFrame(asteroids_list)
    display(df.head())

    df.to_csv("neo_asteroids.csv", index=False) # export dataframe into a csv file
else:
    print(f"Error: Failed to fetch data from NASA NEO API. Status code: {response.status_code}")
    print(f"Response: {response.text}")

,Asteroid ID,Asteroid Name,Minimal Diameter (km),Absolute Magnitude,Relative Velocity (km/s)
0,3150194,(2003 DZ15),0.087610,22.41,8.8979281582
1,3368733,(2007 DG8),0.024241,25.20,9.9229560885
2,3381230,(2007 PB8),0.142087,21.36,22.3674739009
3,3799876,(2018 ES3),0.031373,24.64,8.7353626761
4,3838888,(2019 EO),0.061455,23.18,11.2809750662
